# Train Model – Predicting Employee Performance Rating

**Goal:** build and compare classification models that predict `PerformanceRating` (2 Good, 3 Excellent, 4 Outstanding), and save the best one.

Two models are built because the business wants to use a model **when hiring**:

| Model | Inputs | Use |
|---|---|---|
| **Model A – all factors** | every employee column | understand and monitor current employees |
| **Model B – hiring time factors** | only information available before or at joining | screening new candidates |

Some of the strongest factors (environment satisfaction, last salary hike, years since promotion) only exist **after** a person has joined, so Model B is the honest model for hiring decisions.

**Evaluation approach**
- 80 percent training and 20 percent test data (split keeps the rating proportions)
- 5 fold cross validation on the training data to compare and tune models
- The test data is used **once**, at the end, for the final report
- Macro F1 (average across the three ratings) is the main score because the data is imbalanced

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

ROOT = Path("../..").resolve()
RAW_FILE = ROOT / "data" / "raw" / "INX_Future_Inc_Employee_Performance_CDS_Project2_Data_V1_8.xls"
PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "src" / "models"
FIGURES = ROOT / "src" / "visualization" / "figures"
TARGET = "PerformanceRating"
RANDOM_STATE = 42

import json
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

data = pd.read_csv(PROCESSED / "employee_cleaned.csv")
ALL_FEATURES = [c for c in data.columns if c not in ("EmpNumber", TARGET)]
HIRING_FEATURES = ["Age", "Gender", "EducationBackground", "MaritalStatus", "EmpDepartment", "EmpJobRole",
                   "BusinessTravelFrequency", "DistanceFromHome", "EmpEducationLevel", "NumCompaniesWorked",
                   "TotalWorkExperienceInYears", "EmpJobLevel", "EmpHourlyRate"]
print(len(ALL_FEATURES), "factors in Model A;", len(HIRING_FEATURES), "factors in Model B")

train_index, test_index = train_test_split(data.index, test_size=0.2, stratify=data[TARGET], random_state=RANDOM_STATE)
train, test = data.loc[train_index], data.loc[test_index]
print("Training rows:", len(train), "| Test rows:", len(test))

26 factors in Model A; 13 factors in Model B
Training rows: 960 | Test rows: 240


## 1. Preprocessing pipeline and candidate models

In [2]:
def build_preprocessor(features):
    categorical = [c for c in features if not pd.api.types.is_numeric_dtype(data[c])]
    numeric = [c for c in features if c not in categorical]
    return ColumnTransformer([
        ("categorical", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), categorical),
        ("numeric", StandardScaler(), numeric),
    ])

def candidate_models():
    return {
        "Most frequent class (baseline)": DummyClassifier(strategy="most_frequent"),
        "Logistic Regression": LogisticRegression(max_iter=3000, class_weight="balanced"),
        "Decision Tree": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
        "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "Support Vector Machine": SVC(class_weight="balanced", random_state=RANDOM_STATE),
        "K Nearest Neighbours": KNeighborsClassifier(n_neighbors=15),
    }

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def compare_models(features):
    rows = []
    for name, model in candidate_models().items():
        pipe = Pipeline([("prep", build_preprocessor(features)), ("model", model)])
        scores = cross_validate(pipe, train[features], train[TARGET], cv=cv,
                                scoring={"accuracy": "accuracy", "macro_f1": "f1_macro"}, n_jobs=-1)
        rows.append({"model": name,
                     "cv_accuracy": scores["test_accuracy"].mean(),
                     "cv_macro_f1": scores["test_macro_f1"].mean()})
    return pd.DataFrame(rows).sort_values("cv_macro_f1", ascending=False).round(4).reset_index(drop=True)

## 2. Model A – all factors: compare algorithms

In [3]:
comparison_all = compare_models(ALL_FEATURES)
comparison_all

,model,cv_accuracy,cv_macro_f1
0,Gradient Boosting,0.9312,0.8932
1,Decision Tree,0.8833,0.8317
2,Random Forest,0.9031,0.8212
3,Support Vector Machine,0.7698,0.6998
4,Logistic Regression,0.7365,0.6684
5,K Nearest Neighbours,0.7438,0.4051
6,Most frequent class (baseline),0.7281,0.2809


**Reading the comparison (Model A):** Gradient Boosting is the strongest algorithm (cross validation accuracy about 93 percent and macro F1 about 0.89). The baseline that always predicts rating 3 reaches about 73 percent accuracy but a macro F1 of only 0.28, which shows why macro F1 is the fairer score. Logistic Regression, Support Vector Machine and K Nearest Neighbours are much weaker, which suggests the relationship between the factors and the rating is not simple and linear.

## 3. Model B – hiring time factors: compare algorithms

In [4]:
comparison_hiring = compare_models(HIRING_FEATURES)
comparison_hiring

,model,cv_accuracy,cv_macro_f1
0,Logistic Regression,0.3646,0.3249
1,Support Vector Machine,0.3885,0.3104
2,Gradient Boosting,0.6938,0.3088
3,Random Forest,0.7198,0.2868
4,K Nearest Neighbours,0.7240,0.2839
5,Decision Tree,0.3240,0.2816
6,Most frequent class (baseline),0.7281,0.2809


**Reading the comparison (Model B):** with only hiring time information, no algorithm is meaningfully better than the baseline (macro F1 about 0.28 to 0.32 for all). The information available before joining does not carry enough signal about later performance.

## 4. Tune the two strongest tree based candidates and pick the winner

Random Forest and Gradient Boosting are tuned with a small grid search (5 fold cross validation, macro F1). The winner for each model is chosen from **cross validation only**.

In [5]:
def tune_and_select(features):
    prep = build_preprocessor(features)
    searches = {
        "Random Forest": (RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
                          {"model__n_estimators": [200, 400], "model__max_depth": [None, 8, 12], "model__min_samples_leaf": [1, 2, 4]}),
        "Gradient Boosting": (GradientBoostingClassifier(random_state=RANDOM_STATE),
                              {"model__n_estimators": [100, 200], "model__learning_rate": [0.05, 0.1], "model__max_depth": [2, 3]}),
    }
    results = {}
    for name, (estimator, grid) in searches.items():
        search = GridSearchCV(Pipeline([("prep", prep), ("model", estimator)]), grid, cv=cv, scoring="f1_macro", n_jobs=-1)
        search.fit(train[features], train[TARGET])
        results[name] = search
        print(f"{name}: best cross validation macro F1 = {search.best_score_:.4f} with {search.best_params_}")
    best_name = max(results, key=lambda n: results[n].best_score_)
    return best_name, results[best_name]

best_name_all, best_search_all = tune_and_select(ALL_FEATURES)
print("Selected for Model A:", best_name_all)

Random Forest: best cross validation macro F1 = 0.8914 with {'model__max_depth': None, 'model__min_samples_leaf': 2, 'model__n_estimators': 400}


Gradient Boosting: best cross validation macro F1 = 0.8932 with {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 100}
Selected for Model A: Gradient Boosting


In [6]:
best_name_hiring, best_search_hiring = tune_and_select(HIRING_FEATURES)
print("Selected for Model B:", best_name_hiring)

Random Forest: best cross validation macro F1 = 0.3536 with {'model__max_depth': None, 'model__min_samples_leaf': 4, 'model__n_estimators': 200}


Gradient Boosting: best cross validation macro F1 = 0.3361 with {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 200}
Selected for Model B: Random Forest


## 5. Final evaluation on the untouched test data

In [7]:
def evaluate(search, features, label):
    model = search.best_estimator_
    predicted = model.predict(test[features])
    actual = test[TARGET]
    report = classification_report(actual, predicted, output_dict=True, zero_division=0)
    print(label)
    print(classification_report(actual, predicted, zero_division=0))
    matrix = confusion_matrix(actual, predicted, labels=[2, 3, 4])
    print("Confusion matrix (rows = actual 2, 3, 4; columns = predicted 2, 3, 4)")
    print(matrix, "\n")
    return {"accuracy": accuracy_score(actual, predicted),
            "macro_f1": f1_score(actual, predicted, average="macro"),
            "weighted_f1": f1_score(actual, predicted, average="weighted"),
            "per_class": {k: v for k, v in report.items() if k in ("2", "3", "4")},
            "confusion_matrix": matrix.tolist()}

baseline_accuracy = (test[TARGET] == 3).mean()
print(f"Baseline (always predict rating 3) accuracy: {baseline_accuracy:.3f}\n")
metrics_all = evaluate(best_search_all, ALL_FEATURES, f"MODEL A – all factors ({best_name_all})")
metrics_hiring = evaluate(best_search_hiring, HIRING_FEATURES, f"MODEL B – hiring time factors ({best_name_hiring})")

Baseline (always predict rating 3) accuracy: 0.729

MODEL A – all factors (Gradient Boosting)
              precision    recall  f1-score   support

           2       0.89      0.85      0.87        39
           3       0.94      0.97      0.96       175
           4       0.91      0.77      0.83        26

    accuracy                           0.93       240
   macro avg       0.91      0.86      0.89       240
weighted avg       0.93      0.93      0.93       240

Confusion matrix (rows = actual 2, 3, 4; columns = predicted 2, 3, 4)
[[ 33   5   1]
 [  4 170   1]
 [  0   6  20]] 

MODEL B – hiring time factors (Random Forest)
              precision    recall  f1-score   support

           2       0.28      0.36      0.31        39
           3       0.77      0.80      0.78       175
           4       0.00      0.00      0.00        26

    accuracy                           0.64       240
   macro avg       0.35      0.39      0.37       240
weighted avg       0.60      0.64  

In [8]:
summary = pd.DataFrame({
    "Model A – all factors": [best_name_all, metrics_all["accuracy"], metrics_all["macro_f1"], metrics_all["weighted_f1"]],
    "Model B – hiring time factors": [best_name_hiring, metrics_hiring["accuracy"], metrics_hiring["macro_f1"], metrics_hiring["weighted_f1"]],
}, index=["algorithm", "test_accuracy", "test_macro_f1", "test_weighted_f1"])
summary

,Model A – all factors,Model B – hiring time factors
algorithm,Gradient Boosting,Random Forest
test_accuracy,0.929167,0.641667
test_macro_f1,0.885604,0.365577
test_weighted_f1,0.927791,0.621422


## Conclusions from the model results

**Model A – all factors (Gradient Boosting)**
- Test accuracy about **92.9 percent**, macro F1 about **0.89**, compared with a baseline accuracy of 72.9 percent.
- It recognises all three ratings well: F1 of 0.87 for rating 2, 0.96 for rating 3 and 0.83 for rating 4.
- The main errors are a few employees rated 4 predicted as 3 (recall 77 percent for rating 4). Only 26 test employees have rating 4, so this figure is less stable than the others.
- This model is suitable for **identifying likely lower performers among current employees**, which supports the CEO's need for clear indicators.

**Model B – hiring time factors (Random Forest)**
- Test accuracy about **64 percent**, which is **below the 72.9 percent baseline**, and it never correctly finds a rating 4 employee.
- The honest conclusion is that **candidate information available at hiring (age, education, past experience, role, hourly rate and so on) cannot reliably predict later performance in this data**. Model B is saved for completeness but it is **not recommended for hiring decisions**.
- The factors that drive performance (environment satisfaction, salary hike, promotion timing) are created *after* joining. Model A should therefore be applied after a person has joined and settled in (for example after a probation period), and the same factors should be used to shape the working environment for new joiners.

## 6. Save the trained models and results

In [9]:
MODELS.mkdir(parents=True, exist_ok=True)
joblib.dump(best_search_all.best_estimator_, MODELS / "model_a_all_factors.joblib")
joblib.dump(best_search_hiring.best_estimator_, MODELS / "model_b_hiring_factors.joblib")

metadata = {
    "target": TARGET,
    "classes": [2, 3, 4],
    "class_labels": {"2": "Good", "3": "Excellent", "4": "Outstanding"},
    "random_state": RANDOM_STATE,
    "baseline_accuracy": baseline_accuracy,
    "model_a": {"algorithm": best_name_all, "features": ALL_FEATURES,
                "best_parameters": best_search_all.best_params_, "cv_macro_f1": best_search_all.best_score_, "test": metrics_all},
    "model_b": {"algorithm": best_name_hiring, "features": HIRING_FEATURES,
                "best_parameters": best_search_hiring.best_params_, "cv_macro_f1": best_search_hiring.best_score_, "test": metrics_hiring},
}
with open(MODELS / "model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)

comparison_all.assign(model_set="Model A – all factors").to_csv(PROCESSED / "model_comparison_all_factors.csv", index=False)
comparison_hiring.assign(model_set="Model B – hiring time factors").to_csv(PROCESSED / "model_comparison_hiring_factors.csv", index=False)
print(sorted(p.name for p in MODELS.glob("*")))

['model_a_all_factors.joblib', 'model_b_hiring_factors.joblib', 'model_metadata.json', 'train_model.ipynb']
